# Jones Matrices and Mixed Polarization

### Everything between the sky and the recorded number

Notebook 02 ended with a station recording two feed voltages, on a mount that
turns against the sky, and a baseline turning those into four numbers. That
description was deliberately generous: it assumed the feeds were perfect, the
electronics honest, and both ends of every baseline built the same way.

None of those hold, so this notebook does the accounting properly. Everything a
station does to the signal, the rotation and the imperfect optics and the unknown
electronic gain, collapses into a single $2\times2$ complex matrix. Once that
machinery is in place you can say exactly what goes wrong when the two ends of a
baseline disagree about what they are recording.

**What it covers**

1. why one $2\times2$ matrix per station describes the whole instrument, and
   how it factors into gain, leakage and field rotation;
2. the **measurement equation** that turns a sky into four correlation
   products, and what calibration can and cannot undo;
3. why the same physical rotation looks like a *phase* to a circular-feed
   station and a *mixing* to a linear-feed one;
4. how leakage manufactures polarization that was never in the sky;
5. why the four numbers a baseline records are a **choice of basis**, and what
   breaks when the two ends make different choices.

**Prerequisites:** notebooks 01 and 02, and matrix multiplication. Nothing in
here is bigger than a complex $2\times2$ matrix.

In [1]:
# Every function this notebook calls lives in ../scripts.
import sys
from pathlib import Path

SCRIPTS = Path.cwd() / "scripts"
if not SCRIPTS.is_dir():
    SCRIPTS = Path.cwd().parent / "scripts"
sys.path.append(str(SCRIPTS))
print("importing from", SCRIPTS)

import interactive_jones as explore
import jones
import numpy as np
import polarization as pol

importing from /mnt/c/Users/alexw/OneDrive - University of Toronto/SUMMER_26/EHT/CSC494-report/scripts


## 1. A station is a 2×2 matrix

A station's two feeds each produce a voltage. Whatever happened to the signal
on the way, whether reflected off the dish, rotated by the mount, leaked between
feeds or amplified by an imperfect receiver, the end result is *linear* in the
incoming field. Two outputs, two inputs, linear: that is a $2\times2$ matrix.

$$\begin{pmatrix} v_1 \\ v_2 \end{pmatrix} = \mathbf{J}
\begin{pmatrix} E_x \\ E_y \end{pmatrix}$$

$\mathbf{J}$ is the station's **Jones matrix**, and as far as polarization goes it
is the entire instrument. The usual thing is to factor it into three pieces, each
with a physical name:

$$\mathbf{J} = \underbrace{\mathbf{G}}_{\text{gain}}\;
  \underbrace{(\mathbf{I} + \mathbf{D})}_{\text{leakage}}\;
  \underbrace{\mathbf{\Phi}}_{\text{field rotation}}$$

- $\mathbf{G} = \mathrm{diag}(g_1, g_2)$ holds one unknown complex number per
  feed. Amplitude drifts with the weather, phase drifts with the atmosphere.
- $\mathbf{I} + \mathbf{D} = \begin{pmatrix} 1 & d_1 \\ d_2 & 1\end{pmatrix}$
  holds the **D-terms**, the fraction of each feed's signal that ends up in the
  other one. A few percent is normal.
- $\mathbf{\Phi}$ is the feed frame rotated against the sky, as in notebook 02.

The order matters, and it isn't arbitrary: the mount rotates the sky *before* the
optics leak, and that happens *before* the electronics amplify. Matrices don't
commute, so writing them in the wrong order describes a different telescope.

Build one and look at it:

In [2]:
explore.jones_chain_explorer()

Two things to read off those four panels.

**The gain is diagonal.** It scales and phases each feed separately and never
moves signal between them. That's why station gains, the largest errors in VLBI
by far, can be beaten with closure quantities: they factor out.

**Leakage and field rotation are not diagonal.** They move signal *between*
feeds, which means they move signal between the four correlation products of a
baseline, which means they move signal between $I$, $Q$, $U$ and $V$. No closure
trick removes them. They have to be modelled.

## 2. Two stations make a measurement

A single station's voltages are useless on their own, dominated by noise and
with a meaningless phase. What an interferometer computes is the correlation
between *two* stations, and with two feeds at each end that is a $2\times2$
matrix of products.

The sky needs writing in that shape too. Notebook 01 built the Stokes parameters
out of averaged products of field components: $\langle |E_X|^2\rangle$,
$\langle E_X E_Y^*\rangle$, and so on. Arrange those same four averages in a
square instead of a list and the result is the **coherency matrix**:

$$\mathbf{C} = \langle \mathbf{E}\mathbf{E}^\dagger \rangle =
\begin{pmatrix} \langle E_X E_X^*\rangle & \langle E_X E_Y^*\rangle \\
                \langle E_Y E_X^*\rangle & \langle E_Y E_Y^*\rangle
\end{pmatrix}$$

It carries exactly what $I, Q, U, V$ carry, in the shape the instrument acts on.
A fixed $4\times4$ matrix converts between the two, and Section 5 is entirely
about that matrix. ($\mathbf{E}^\dagger$ is the *conjugate transpose*: turn the
column into a row and flip the sign of every imaginary part. For a single number
it is just the complex conjugate.)

With the sky in that form, the four numbers a baseline records are

$$\mathbf{V}_{\rm obs} = \mathbf{J}_1\, \mathbf{C}\, \mathbf{J}_2^\dagger$$

That's the **radio interferometer measurement equation**, and every serious
calibration and imaging package is a way of solving it. It says the corruption
is *station based*: station 1 acts on the left, station 2 on the right, and each
appears in every baseline it touches.

The four entries of $\mathbf{V}_{\rm obs}$ are the four numbers written to disk,
so the obvious question is how badly a realistic instrument damages them.
Drag the leakage up from zero:

In [3]:
explore.rime_explorer()

Stokes $I$ barely moves. $Q$ and $U$ are wrecked by a few percent of leakage,
because they are small *differences* of large numbers, and $V$, smaller still, is
worse. That asymmetry is why polarimetry needs calibration an order of magnitude better
than total-intensity imaging does.

Notice what *isn't* the problem: if $\mathbf{J}_1$ and $\mathbf{J}_2$ are known,
the equation inverts exactly,

$$\mathbf{C} = \mathbf{J}_1^{-1}\, \mathbf{V}_{\rm obs}\,
\mathbf{J}_2^{-\dagger}$$

and the sky comes back to machine precision. Calibration isn't hard because the
algebra is hard. It's hard because those matrices are unknown, they drift through
the night, and they have to be inferred from the same data you are trying to
interpret.

In [4]:
truth = pol.stokes_from_ellipse(1.0, 0.3, np.deg2rad(30.0), 0.05)
station1 = jones.jones_matrix(1.3 + 0.1j, 0.8, 0.05, -0.02, "rl", np.deg2rad(15))
station2 = jones.jones_matrix(0.9, 1.1 - 0.3j, -0.03, 0.04, "rl", np.deg2rad(-40))

clean = jones.baseline_coherency(truth, "rl", "rl")
observed = jones.apply_jones(station1, clean, station2)

# Undo the (known) corruption, then convert with this baseline's own matrix.
repaired = jones.recover_coherency(observed, station1, station2)
recovered = jones.stokes_from_slots(jones.slots_of(repaired), "rl", "rl")

print("on the sky       ", np.round(truth, 6))
print("read uncorrected ", np.round(jones.stokes_from_slots(
    jones.slots_of(observed), "rl", "rl"), 6))
print("after calibration", np.round(recovered, 6))
print(f"largest error     {np.abs(recovered - truth).max():.1e}")

on the sky        [1.       0.15     0.259808 0.05    ]
read uncorrected  [ 0.652989  0.244795  0.251147 -0.019832]
after calibration [1.       0.15     0.259808 0.05    ]
largest error     5.6e-17


## 3. One rotation, two different corrections

Now the first place the two feed types genuinely part ways.

The field rotation $\mathbf{\Phi}$ has one definition, and it doesn't care what
feeds a station has: rotate the sky frame by the angle, then write that rotation
in the station's own feed basis,

$$\mathbf{\Phi} = \mathbf{F}\, \mathbf{R}(\phi)\, \mathbf{F}^{-1}$$

where $\mathbf{R}$ is an ordinary $2\times2$ rotation by $\phi$ and $\mathbf{F}$
is the station's **feed matrix**: the change of basis from the field's $X$ and $Y$
components to whatever its own two feeds respond to. For a linear-feed station
that is the identity, since its feeds *are* $X$ and $Y$, and for a circular-feed
station it is notebook 01's $R, L \leftarrow X, Y$ matrix. One definition, no
special cases. Watch what comes out of it for the two kinds of station:

In [5]:
explore.field_rotation_explorer()

For **circular** feeds the result is diagonal: a phase $e^{-i\phi}$ on one feed
and $e^{+i\phi}$ on the other. The amplitudes $|R|$ and $|L|$ never change.

For **linear** feeds it's a real rotation with off-diagonal entries: the feeds
get mixed into each other, so $|X|$ and $|Y|$ *do* change.

Both describe the same rotation of the same sky. In Stokes terms both rotate the
$(Q, U)$ vector by $2\phi$, or equivalently turn the measured EVPA by $\phi$,
exactly as notebook 01's factor of one half implies. But the *correction* a
station has to apply is a phase in one case and a mixing in the other, and a
pipeline written for one has no code path for the other.

In [6]:
angle = np.deg2rad(30.0)
for feeds in ("rl", "xy"):
    phi = jones.field_rotation_matrix(feeds, angle)
    quiet = jones.observe(truth, feeds, feeds)
    turned = jones.observe(truth, feeds, feeds, phi, phi)
    labels = jones.slot_labels(feeds, feeds)
    print(f"{feeds.upper()} feeds, parallel-hand amplitudes "
          f"|{labels[0]}|, |{labels[1]}|:")
    print(f"   at rest  {np.abs(quiet[:2]).round(4)}")
    print(f"   rotated  {np.abs(turned[:2]).round(4)}"
          f"   {'unchanged' if np.allclose(np.abs(quiet[:2]), np.abs(turned[:2])) else 'CHANGED'}")

RL feeds, parallel-hand amplitudes |RR|, |LL|:
   at rest  [1.05 0.95]
   rotated  [1.05 0.95]   unchanged
XY feeds, parallel-hand amplitudes |XX|, |YY|:
   at rest  [1.15 0.85]
   rotated  [0.85 1.15]   CHANGED


## 4. Leakage manufactures polarization

The D-terms deserve their own section, because of how they fail.

Leakage puts a little of feed 2's signal into feed 1. Of a baseline's four
products, two pair a feed with the same kind of feed at the far end. Those are
the **parallel hands**, RR and LL on a circular-feed array, and the other two
pair opposite kinds and are the **cross hands**, RL and LR. On a circular-feed
array the cross hands are where $Q$ and $U$ live, so on an unpolarized source
they should read zero. Leakage fills them in anyway, by an amount proportional to
the D-terms, and both ends of the baseline contribute, so

$$p_{\rm false} \approx |d_1 + d_2^*|$$

and 1% leakage at each station invents about 2% polarization. Set the source's
own polarization to **zero** and drag the leakage up. Every tick that appears
is the instrument:

In [8]:
explore.leakage_explorer()

What makes this dangerous isn't the size but the *shape*. The false signal is
smooth, coherent across the image, and looks exactly like a real polarization
pattern, with nothing about it resembling noise. Since M87's circular
polarization is a few tenths of a percent, uncorrected leakage of a few percent
doesn't look like a bug in the pipeline. It looks like a result.

This is why D-terms are solved for as part of calibration, and why the EHT's
polarimetric papers spend so much of their length on them.

## 5. The four numbers are a choice of basis

Everything so far has been about *corrupting* the four correlation products.
Now the deeper question: what are those four numbers to begin with?

A baseline correlates each feed of station 1 against each feed of station 2. So
the four products are labelled by a *pair* of feeds, one from each end, and their
names depend on both:

In [9]:
explore.slot_explorer()

Two circular stations give the familiar RR, LL, RL, LR. Two linear stations give
XX, YY, XY, YX. One of each gives **XR, YL, XL, YR**, four products that belong to
neither convention.

The four numbers always sit in the same four positions in the file, whatever they
happen to mean. Those positions are what the code calls **slots**, and the name
belonging to a slot is a fact about the two stations, not about the file.

eht-imaging records the pairing as a per-baseline string it calls `polbasis`
(`'rlrl'`, `'xyxy'`, `'rlxy'`, …), alongside the older `polrep`, a single label
saying which basis an *entire* dataset is in. A dataset whose rows all share one
pairing is *homogeneous*; one carrying several is what `polrep='mixed'` exists to
describe.

Why does it matter? Because turning the four products into $I$, $Q$, $U$, $V$ is
a linear map, and **the map depends on the pairing**.

Watch the change of size here. The $2\times2$ matrices of Sections 1 and 2 act on
the field's two components, one station at a time, which is the hardware
corrupting the signal. This map does a different job: four recorded products go
in and four Stokes parameters come out, so the thing doing the work is
$4\times4$. Those products are the entries of $\mathbf{C}$ read out as a list
rather than as a square, which is how the same information fits either shape:

In [10]:
explore.conversion_explorer()

Compare the two homogeneous cases. Circular feeds put $I$ and $V$ on the
parallel hands and $Q$, $U$ on the cross hands; linear feeds do the opposite,
with $I$ and $Q$ on the parallel hands. Both are clean, and in either case *one*
matrix serves the entire dataset: convert once, then forget about feeds.

The mixed case has no such split: every Stokes parameter draws on every slot.
And it is a *different* matrix from either homogeneous case, so it can't be
applied globally. It has to be looked up row by row.

That single sentence is the software problem. Everything else follows from it.

## 6. What breaks when the ends disagree

Suppose a pipeline written for a homogeneous circular array is handed data from
an array containing ALMA. Nothing crashes. The four numbers are there; the code
multiplies them by the $4\times4$ matrix for the basis it *believes* was used,
and out comes a Stokes vector.

Here is what that vector looks like:

In [11]:
explore.misread_explorer()

Try the **all linear read as circular** preset with a purely linearly polarized
source: the entire signal lands in Stokes $V$. Real circular polarization in M87
is at the few-tenths-of-a-percent level, so a result like that isn't obviously an
error. It is a spectacular and completely fictitious detection.

The mixed preset is worse still. There, not even the total intensity survives: in
the table below a 1 Jy source is reported at about half a jansky, with $Q$, $U$
and $V$ scrambled by amounts that depend on the source itself. There's no
residual, no failed fit, no warning. The numbers are just wrong.

In [12]:
cases = {
    "all circular, read correctly": (("rl", "rl"), ("rl", "rl")),
    "all linear, read as circular": (("xy", "xy"), ("rl", "rl")),
    "ALMA × SMA, read as circular": (("xy", "rl"), ("rl", "rl")),
    "ALMA × SMA, read correctly": (("xy", "rl"), ("xy", "rl")),
}
print(f"{'':32s}     I      Q      U      V")
print(f"{'on the sky':32s} {truth[0]:+.3f} {truth[1]:+.3f} {truth[2]:+.3f} {truth[3]:+.3f}")
for label, (real_feeds, assumed) in cases.items():
    out = jones.misread_stokes(truth, real_feeds, assumed)
    flag = "" if np.allclose(out, truth, atol=1e-9) else "   <- wrong"
    print(f"{label:32s} {out[0]:+.3f} {out[1]:+.3f} {out[2]:+.3f} {out[3]:+.3f}{flag}")

                                     I      Q      U      V
on the sky                       +1.000 +0.150 +0.260 +0.050
all circular, read correctly     +1.000 +0.150 +0.260 +0.050
all linear, read as circular     +1.000 +0.260 +0.050 +0.150   <- wrong
ALMA × SMA, read as circular     +0.516 +0.481 +0.410 +0.332   <- wrong
ALMA × SMA, read correctly       +1.000 +0.150 +0.260 +0.050


### What has to change in the software

Three concrete things, all of them consequences of Section 5:

1. **The polarization basis has to be a property of a row, not of a dataset.**
   A single global `polrep` can't describe an array where baselines differ, so
   the data model needs a per-station `feed_type` and a per-row `polbasis`, and
   the four slots need neutral names, `p1p1`, `p2p2`, `p1p2` and `p2p1`, instead
   of `RR`, `LL`, `RL`, `LR`.
2. **Every conversion has to be looked up per baseline.** Anywhere the old code
   multiplied a whole dataset by one $4\times4$ matrix, it now has to fetch the
   matrix for that row's pairing.
3. **Noise pairs by feed, not by station.** From notebook 02: $\sigma$ for a
   product is set by the two *feeds* that formed it, so a mixed baseline can
   carry four different error bars. Code that hard-codes an R/L pairing gets
   them silently wrong.

The same three points apply to what is written to disk. Interferometry data is
exchanged as **uvfits** files, and a uvfits file describes its polarization axis
with one number for the whole file, `CRVAL3`, which is exactly the assumption that
just broke. What actually identifies a mixed file is the per-station feed
tags, `POLTYA`/`POLTYB`; with those missing, a mixed dataset is indistinguishable
from a circular one.

## 7. Checking it against the real thing

Every convention in this notebook is a chance to be subtly wrong: the sign of
$V$, which slot is `p1p2`, whether $\mathbf{\Phi}$ multiplies on the left or the
right. The only defence is checking it against the code that has to agree with it.

The comparison below is against `eht-imaging`'s `pol_conventions` module, for all
four feed pairings. What gets compared is the *matrices*, not one sample sky.
Both directions of the Stokes conversion are linear, so agreeing on the
$4\times4$ matrix means agreeing for every possible source at once.

Those matrices are read out of `eht-imaging` by
`scripts/export_ehtim_reference.py` and committed to
`scripts/data/ehtim_pol_conventions.json`, because these notebooks run in an
environment that deliberately has no `eht-imaging` in it. The last line re-reads
the live module wherever there is one and says whether the committed file still
matches, so a stale capture gets caught instead of believed.

In [13]:
import ehtim_reference

reference = ehtim_reference.load()
print(f"against eht-imaging v{reference['ehtim_version']}, "
      f"{reference['ehtim_module']}, captured {reference['captured']}\n")

for label, error in ehtim_reference.comparisons(reference):
    verdict = "agree" if error < ehtim_reference.TOLERANCE else "DIFFER"
    print(f"{verdict}   {label:34s} max difference {error:.1e}")

print(f"\n{ehtim_reference.check_live(reference)}")

against eht-imaging v1.3.0, eht-imaging/ehtim/observing/pol_conventions.py, captured 2026-08-23

agree   feed matrix RL                     max difference 0.0e+00
agree   feed matrix XY                     max difference 0.0e+00
agree   Stokes → slots, RL×RL              max difference 0.0e+00
agree   slots → Stokes, RL×RL              max difference 0.0e+00
agree   Stokes → slots, XY×XY              max difference 0.0e+00
agree   slots → Stokes, XY×XY              max difference 0.0e+00
agree   Stokes → slots, XY×RL              max difference 0.0e+00
agree   slots → Stokes, XY×RL              max difference 0.0e+00
agree   Stokes → slots, RL×XY              max difference 0.0e+00
agree   slots → Stokes, RL×XY              max difference 0.0e+00
agree   G · (I + D)                        max difference 0.0e+00

eht-imaging is not installed in this environment, so the comparison above uses the committed capture; scripts/tests/test_ehtim_reference.py redoes it against the live module wh

## Recap

- Everything a station does to the signal is one $2\times2$ complex matrix,
  $\mathbf{J} = \mathbf{G}(\mathbf{I}+\mathbf{D})\mathbf{\Phi}$, and the factors
  don't commute.
- A baseline obeys the measurement equation
  $\mathbf{V}_{\rm obs} = \mathbf{J}_1 \mathbf{C}\mathbf{J}_2^\dagger$. Gains are
  diagonal and factor out of closure quantities; leakage and field rotation are
  not, and have to be modelled.
- Field rotation is **one** definition, $\mathbf{F}\mathbf{R}\mathbf{F}^{-1}$,
  that looks like a phase to circular feeds and a mixing to linear ones.
- Leakage **invents** polarization: 1% at each end gives about 2%, smooth and
  plausible, in a regime where the real signal is a few tenths of a percent.
- The four numbers a baseline records are labelled by a *pair* of feeds. The
  $4\times4$ conversion to $I, Q, U, V$ depends on that pairing, so in a mixed
  array it is a per-row lookup rather than one global matrix, and a pipeline that
  assumes otherwise reports wrong Stokes parameters with no warning.

**Next:** notebook 04, which puts the forward model to work: simulate a mixed-feed
observation end to end, then reconstruct a polarized image from it.